# Assignment 2. Urban Flooding Modeling

Please download DEM in Pennsylvania and run the HAND model

Please download the DEM tiles for Pennsylvania

Run the HAND model to estimate the inundation depth for each tile

Export the results to geotiff

Mosaic the exposed results into a complete geotiff for whole Pennsylvania.

Upload your notebook and submit a report to present your map for PA.

In [ ]:
def get_tile_name(lat, lon):
    """
    Returns the USGS tile name formatted correctly (e.g., 'n39w076').
    """
    lat_tile = f"n{int(abs(lat)):02d}" if lat >= 0 else f"s{int(abs(lat)):02d}"
    lon_tile = f"w{int(abs(lon)):03d}" if lon < 0 else f"e{int(abs(lon)):03d}"
    return f"{lat_tile}{lon_tile}"

# Example: Get tile for Philadelphia (39.95°N, 75.16°W)
lat, lon = 39.95, -75.16
tile_name = get_tile_name(lat, lon)
print("DEM Tile Name:", tile_name)

In [ ]:
def get_tiles_in_bbox(min_lat, max_lat, min_lon, max_lon):
    """
    Returns a list of tile names covering a bounding box.
    """
    tile_list = []
    for lat in range(int(min_lat), int(max_lat) + 1):
        for lon in range(int(min_lon), int(max_lon) + 1):
            tile_list.append(get_tile_name(lat, lon))
    return tile_list

# Example: Get tiles for an area (Philadelphia to NYC region)
tiles = get_tiles_in_bbox(39, 43, -81, -74)
print("Tiles to Download:", tiles)


In [ ]:
import requests
import os

def download_dem_1arc(tile_name, save_path="."):
    """
    Downloads a 1-arc-second (~30m) DEM tile from USGS TNM.
    
    Parameters:
        tile_name (str): Tile name (e.g., "n39w076").
        save_path (str): Directory to save the DEM file.
    """
    base_url = "https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/1/TIFF/current"
    url = f"{base_url}/{tile_name}/USGS_1_{tile_name}.tif"
    
    output_file = os.path.join(save_path, f"{tile_name}.tif")
    
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(output_file, "wb") as file:
            for chunk in response.iter_content(1024):
                file.write(chunk)
        print(f"✅ DEM Tile {tile_name} downloaded successfully!")
    else:
        print(f"❌ Error: Tile {tile_name} not found. Check availability.")
        print(url)
# Download multiple tiles
save_directory = "DEM_Tiles_PA"
os.makedirs(save_directory, exist_ok=True)  # Create folder if it doesn't exist

for tile in tiles:
    download_dem_1arc(tile, save_directory)

In [ ]:
import numpy as np
import pysheds
from pysheds.grid import Grid
from matplotlib import pyplot as plt
import matplotlib.colors as colors
import geopandas as gpd
from rasterio.mask import mask


Mosaic the exposed results into a complete geotiff for whole Pennsylvania.

In [ ]:
from pathlib import Path
from contextlib import ExitStack
import rasterio
from rasterio.merge import merge

tile_folder = Path("DEM_Tiles_PA")
tif_files = list(tile_folder.glob("*.tif"))

if not tif_files:
    raise FileNotFoundError(f"No .tif files found in {tile_folder}")

with ExitStack() as stack:
    sources = [stack.enter_context(rasterio.open(path)) for path in tif_files]

    mosaic, transform = merge(sources)

    metadata = sources[0].meta.copy()
    metadata.update(
        driver="GTiff",
        height=mosaic.shape[1],
        width=mosaic.shape[2],
        transform=transform,
        compress="lzw",
    )

    with rasterio.open("PA_DEM_mosaic.tif", "w", **metadata) as output:
        output.write(mosaic)

print("Saved PA_DEM_mosaic.tif")

### Mask

In [ ]:
mosaic_file = "PA_DEM_mosaic.tif"
boundary_file = "stateboundary\stateboundary.shp"
output_file = "PA_DEM_mosaic_masked.tif"

boundary = gpd.read_file(boundary_file)

with rasterio.open(mosaic_file) as src:
    boundary = boundary.to_crs(src.crs)

    nodata = src.nodata if src.nodata is not None else -9999
    clipped, transform = mask(
        src, boundary.geometry, crop=True, nodata=nodata
    )

    metadata = src.meta.copy()
    metadata.update(
        height=clipped.shape[1],
        width=clipped.shape[2],
        transform=transform,
        nodata=nodata,
        compress="lzw",
    )

with rasterio.open(output_file, "w", **metadata) as dest:
    dest.write(clipped)

print(f"Saved {output_file}")

### Computing the height above nearest drainage
computing the flow directions and accumulation for a given DEM

In [ ]:
# Instantiate grid from raster
grid = Grid.from_raster('PA_DEM_mosaic_masked.tif')
dem = grid.read_raster('PA_DEM_mosaic_masked.tif')

# # Resolve flats and compute flow directions
# inflated_dem = grid.resolve_flats(dem)
# fdir = grid.flowdir(inflated_dem)

# # Compute accumulation
# acc = grid.accumulation(fdir)

In [ ]:
# Condition DEM
# ----------------------
# Fill pits in DEM

pit_filled_dem = grid.fill_pits(dem)

# Fill depressions in DEM
flooded_dem = grid.fill_depressions(pit_filled_dem)

# Resolve flats in DEM
inflated_dem = grid.resolve_flats(flooded_dem)

In [ ]:
# Determine D8 flow directions from DEM
# ----------------------
# Specify directional mapping
dirmap = (64, 128, 1, 2, 4, 8, 16, 32)
    
# Compute flow directions
# -------------------------------------
fdir = grid.flowdir(inflated_dem, dirmap=dirmap)

In [ ]:
if not hasattr(np, "in1d"):
    np.in1d = np.isin
    
acc = grid.accumulation(fdir, dirmap=dirmap)

In [ ]:
# Compute height above nearest drainage
hand = grid.compute_hand(fdir, dem, acc > 200)

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_alpha(0)
plt.imshow(hand, extent=grid.extent, cmap='terrain', zorder=1)
plt.colorbar(label='Height above nearest drainage (m)')
plt.grid(zorder=0)
plt.title('HAND', size=14)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()

In [ ]:
# Create a view of HAND in the catchment
hand_view = grid.view(hand, nodata=np.nan)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
cmap = plt.get_cmap("terrain").copy()
cmap.set_bad("#f4f6f9")  # background for masked cells

im = ax.imshow(
    hand,
    extent=grid.extent,
    origin="upper",
    cmap=cmap,
    vmin=0,
    vmax=100,
    interpolation="nearest",
    aspect="auto"
)

pa.boundary.plot(ax=ax, color="#263746", linewidth=0.8)

fig.colorbar(
    im, ax=ax,
    label="Height above nearest drainage (m)",
    extend="max"
)
ax.set(xlabel="Longitude", ylabel="Latitude", title="HAND")
ax.set_xlim(-80.55, -74.65)
ax.set_ylim(39.7, 42.3)
ax.grid(zorder=0, alpha=0.35)
fig.tight_layout()

### Estimating inundation extent (constant channel depth)
We can estimate the inundation extent (assuming a constant channel depth) using a simple binary threshold:

Explanation of the function numpy.where(condition, x, y),

condition: A boolean condition that is evaluated on the array. x (optional): Values to select when the condition is True. y (optional): Values to select when the condition is False.

In [ ]:
threshold = 3

#if the hand value is larger than threshold, then use nan
inundation_extent = np.where(hand_view < threshold, threshold - hand_view, np.nan)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_alpha(0)

dem_view = grid.view(
    dem,
    nodata=np.nan,
    dtype=np.float64
)

ax.imshow(
    dem_view,
    extent=grid.extent,
    cmap="Greys",
    zorder=1
)

ax.imshow(
    inundation_extent,
    extent=grid.extent,
    cmap="Blues",
    vmin=-5,
    vmax=10,
    zorder=2
)

ax.grid(zorder=0)
ax.set_title("Inundation depths (constant channel depth)", size=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.tight_layout()
plt.show()

Export the generated results into a geotiff

In [ ]:
import rasterio as rio

datasource = rio.open('PA_DEM_mosaic_masked.tif')
meta = datasource.meta

# Update metadata if needed (e.g., change datatype, compression, etc.)
meta.update(dtype=rio.float32, compress='lzw', nodata=np.nan)

# Write the output GeoTIFF
with rio.open('out_res.tif', 'w', **meta) as dst:
    dst.write(hand, 1)

## The report

The latest GeoTIFF shows areas at risk of flooding. Darker areas indicate greater flood risk based on how deeply the water may cover the land.